# EG-LightEncoder: Quantum-Inspired Lightweight Surveillance Anomaly Detection

End-to-end PyTorch implementation for UCF-Crime: video-level training, uniform temporal sampling, EfficientNet-Lite0 + GhostNet spatial features, classical quantum-inspired mapping, confidence-adaptive temporal verification, fuzzy alert prioritisation, edge optimisation, and evaluation.

**Important protocol notes**
- Use the official UCF-Crime anomaly-detection train/test split. A validation subset is created from the official training partition only.
- UCF-Crime training labels are video-level weak labels. The notebook therefore uses a video-level multiple-instance objective; it does **not** pretend that every sampled frame in an anomalous training video has a frame-level ground-truth label.
- Test temporal annotations are used only for frame-level evaluation.
- The quantum-inspired mapping is implemented with ordinary PyTorch operations. It does not use a quantum computer or claim quantum advantage.
- Download the dataset and annotation files from the official CRCV project page and place them in the paths configured below. The notebook does not bypass access restrictions or redistribute videos.

## 1. Environment
Run this cell once. A CUDA runtime is recommended for training. The notebook uses `timm` to obtain EfficientNet-Lite0 and GhostNet implementations.

In [ ]:
%pip -q install timm opencv-python-headless scikit-learn pandas matplotlib seaborn tqdm

## 2. Imports and reproducibility

In [ ]:
import os, re, glob, time, math, random, copy, warnings, json
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix,
    classification_report
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## 3. Dataset paths and experiment configuration

In [ ]:
# Update these paths to match the extracted official UCF-Crime directory.
DATA_ROOT = Path("/content/UCF_Crimes")
VIDEO_ROOT = DATA_ROOT / "Videos"
SPLIT_ROOT = DATA_ROOT / "Anomaly_Detection_splits"
TRAIN_SPLIT = SPLIT_ROOT / "Anomaly_Train.txt"
TEST_SPLIT = SPLIT_ROOT / "Anomaly_Test.txt"

# Temporal_Anomaly_Annotation_for_Testing_Videos.txt is commonly supplied
# alongside the split files. Set this path to its actual location.
TEMPORAL_ANNOTATION_FILE = DATA_ROOT / "Temporal_Anomaly_Annotation_for_Testing_Videos.txt"

IMAGE_SIZE = 224
NUM_FRAMES = 16                 # Uniformly sampled frames per video
BATCH_SIZE = 2                  # Number of videos per batch; lower if GPU memory is limited
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2
VAL_SIZE = 0.15
TOPK_RATIO = 0.25
FEATURE_DIM = 256
QUANTIZE_AFTER_TRAINING = True

# For a quick pipeline smoke test, set MAX_TRAIN_VIDEOS to a small integer.
# Keep None for the complete official training partition.
MAX_TRAIN_VIDEOS = None
MAX_TEST_VIDEOS = None

assert VIDEO_ROOT.exists(), f"Video directory not found: {VIDEO_ROOT}"
assert TRAIN_SPLIT.exists(), f"Official train split not found: {TRAIN_SPLIT}"
assert TEST_SPLIT.exists(), f"Official test split not found: {TEST_SPLIT}"

## 4. Read official split files and resolve video paths

The parser supports common split-line forms such as `Abuse/video.mp4 1`, `video.mp4 1`, and path-only entries. When a label is not present, it is inferred from the official video folder/name convention: normal folders are label 0; anomaly-category folders are label 1.

In [ ]:
NORMAL_TOKENS = {
    "normal_videos_event", "testing_normal_videos_anomaly",
    "training_normal_videos_anomaly", "normal"
}

def infer_label_from_text(value):
    s = str(value).replace("\\", "/").lower()
    parts = [p for p in s.split("/") if p]
    if any(p in NORMAL_TOKENS for p in parts):
        return 0
    if "normal" in Path(s).name.lower():
        return 0
    return 1

def build_video_index(video_root):
    index = {}
    for p in Path(video_root).rglob("*"):
        if p.is_file() and p.suffix.lower() in {".mp4", ".avi", ".mov", ".mkv"}:
            index[p.name.lower()] = p
            index[str(p.relative_to(video_root)).replace("\\", "/").lower()] = p
    return index

VIDEO_INDEX = build_video_index(VIDEO_ROOT)
print("Indexed videos:", len(VIDEO_INDEX))

def resolve_video_path(token, video_root=VIDEO_ROOT, index=VIDEO_INDEX):
    token = token.strip().strip('"').strip("'").replace("\\", "/")
    candidate = Path(token)
    if candidate.is_absolute() and candidate.exists():
        return candidate
    direct = video_root / token
    if direct.exists():
        return direct
    key = token.lower().lstrip("./")
    if key in index:
        return index[key]
    name = Path(token).name.lower()
    if name in index:
        return index[name]
    return None

def read_split_file(split_path):
    records, missing = [], []
    with open(split_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            parts = line.split()
            token = parts[0]
            label = None
            if len(parts) > 1 and parts[1] in {"0", "1"}:
                label = int(parts[1])
            path = resolve_video_path(token)
            if path is None:
                missing.append(token)
                continue
            if label is None:
                label = infer_label_from_text(token)
            records.append({"path": str(path), "label": int(label), "video": path.name})
    df = pd.DataFrame(records).drop_duplicates("path").reset_index(drop=True)
    if missing:
        print(f"Unresolved entries in {Path(split_path).name}: {len(missing)}")
        print("Examples:", missing[:5])
    return df

train_all = read_split_file(TRAIN_SPLIT)
test_df = read_split_file(TEST_SPLIT)

print("Resolved train:", len(train_all), train_all.label.value_counts().to_dict())
print("Resolved test :", len(test_df), test_df.label.value_counts().to_dict())
assert len(train_all) > 0 and len(test_df) > 0, "No videos resolved. Check VIDEO_ROOT and split-file paths."

if MAX_TRAIN_VIDEOS is not None:
    train_all = train_all.groupby("label", group_keys=False).head(max(1, MAX_TRAIN_VIDEOS // 2)).reset_index(drop=True)
if MAX_TEST_VIDEOS is not None:
    test_df = test_df.head(MAX_TEST_VIDEOS).reset_index(drop=True)

train_df, val_df = train_test_split(
    train_all, test_size=VAL_SIZE, random_state=SEED, stratify=train_all["label"]
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Video-level split: train={len(train_df)}, validation={len(val_df)}, test={len(test_df)}")
assert set(train_df.path).isdisjoint(set(val_df.path))
assert set(train_df.path).isdisjoint(set(test_df.path))
assert set(val_df.path).isdisjoint(set(test_df.path))

## 5. Video decoding, uniform temporal sampling, resizing, normalization and augmentation

In [ ]:
def read_uniform_frames(video_path, num_frames=NUM_FRAMES, image_size=IMAGE_SIZE):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0
    if total <= 0:
        cap.release()
        raise RuntimeError(f"Video has no readable frames: {video_path}")

    frame_indices = np.linspace(0, max(total - 1, 0), num_frames).round().astype(int)
    frames = []
    last_good = None
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            frame = last_good.copy() if last_good is not None else np.zeros((image_size, image_size, 3), np.uint8)
        else:
            last_good = frame
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        frames.append(frame)
    cap.release()
    return np.stack(frames), frame_indices, fps, total

class VideoTransform:
    def __init__(self, train=False):
        self.train = train
    def __call__(self, frames):
        x = frames.astype(np.float32) / 255.0
        if self.train:
            if random.random() < 0.5:
                x = x[:, :, ::-1, :].copy()
            if random.random() < 0.5:
                factor = random.uniform(0.85, 1.15)
                x = np.clip(x * factor, 0, 1)
        # ImageNet normalization expected by pretrained timm backbones.
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        x = (x - mean) / std
        return torch.from_numpy(x).permute(0, 3, 1, 2).contiguous()

class UCFVideoDataset(Dataset):
    def __init__(self, dataframe, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = VideoTransform(train=train)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        frames, frame_indices, fps, total = read_uniform_frames(row.path)
        frames = self.transform(frames)
        return {
            "frames": frames,
            "label": torch.tensor(float(row.label), dtype=torch.float32),
            "path": row.path,
            "frame_indices": torch.tensor(frame_indices, dtype=torch.long),
            "fps": torch.tensor(fps, dtype=torch.float32),
            "total_frames": torch.tensor(total, dtype=torch.long)
        }

def video_collate(batch):
    return {
        "frames": torch.stack([b["frames"] for b in batch]),
        "label": torch.stack([b["label"] for b in batch]),
        "path": [b["path"] for b in batch],
        "frame_indices": torch.stack([b["frame_indices"] for b in batch]),
        "fps": torch.stack([b["fps"] for b in batch]),
        "total_frames": torch.stack([b["total_frames"] for b in batch])
    }

train_ds = UCFVideoDataset(train_df, train=True)
val_ds = UCFVideoDataset(val_df, train=False)
test_ds = UCFVideoDataset(test_df, train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                          collate_fn=video_collate, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                        collate_fn=video_collate)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
                         collate_fn=video_collate)

sample = train_ds[0]
print("Sample video tensor:", tuple(sample["frames"].shape), "label:", sample["label"].item())

## 6. EG-LightEncoder model

**Contribution 1:** EfficientNet-Lite0 and GhostNet feature maps are resized to a common spatial grid, concatenated, and globally pooled.

**Contribution 2:** normalized pooled features are angle-encoded and passed through parameterized sine/cosine rotations and pairwise interactions. This is a classical feature map.

**Contribution 3:** a TinyMLP estimates frame anomaly probabilities. High-entropy video predictions activate a lightweight temporal convolution; temporal smoothing and fuzzy rules produce alert severity.

In [ ]:
class HybridSpatialEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        try:
            self.efficient = timm.create_model(
                "tf_efficientnet_lite0", pretrained=pretrained,
                features_only=True, out_indices=(-1,)
            )
            self.ghost = timm.create_model(
                "ghostnet_100", pretrained=pretrained,
                features_only=True, out_indices=(-1,)
            )
        except Exception as e:
            print("Pretrained/model download unavailable; using randomly initialized backbones.", repr(e))
            self.efficient = timm.create_model(
                "tf_efficientnet_lite0", pretrained=False,
                features_only=True, out_indices=(-1,)
            )
            self.ghost = timm.create_model(
                "ghostnet_100", pretrained=False,
                features_only=True, out_indices=(-1,)
            )
        self.eff_channels = self.efficient.feature_info.channels()[-1]
        self.ghost_channels = self.ghost.feature_info.channels()[-1]
        self.pool_grid = (7, 7)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.output_dim = self.eff_channels + self.ghost_channels

    def forward(self, x):
        ef = self.efficient(x)[-1]
        gf = self.ghost(x)[-1]
        ef = F.adaptive_avg_pool2d(ef, self.pool_grid)
        gf = F.adaptive_avg_pool2d(gf, self.pool_grid)
        fused = torch.cat([ef, gf], dim=1)
        return self.gap(fused).flatten(1)

class QuantumInspiredMapper(nn.Module):
    def __init__(self, input_dim, output_dim=FEATURE_DIM):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.angle_scale = nn.Parameter(torch.ones(input_dim))
        self.rotation = nn.Linear(input_dim, input_dim)
        self.projection = nn.Sequential(
            nn.Linear(input_dim * 4, output_dim),
            nn.GELU(),
            nn.LayerNorm(output_dim)
        )
    def forward(self, x):
        x = self.norm(x)
        x = torch.tanh(x) * (math.pi / 2.0)
        theta = x * self.angle_scale
        rot = self.rotation(theta)
        # Classical angle encoding and parameterized rotation-inspired mapping.
        sin_a, cos_a = torch.sin(theta), torch.cos(theta)
        sin_r, cos_r = torch.sin(rot), torch.cos(rot)
        # Lightweight nonlinear feature interaction.
        interaction = sin_a * cos_r
        mapped = torch.cat([sin_a, cos_a, sin_r, interaction], dim=1)
        return self.projection(mapped)

class EGLightEncoder(nn.Module):
    def __init__(self, pretrained=True, feature_dim=FEATURE_DIM):
        super().__init__()
        self.spatial = HybridSpatialEncoder(pretrained=pretrained)
        self.quantum_map = QuantumInspiredMapper(self.spatial.output_dim, feature_dim)
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 128), nn.ReLU(inplace=True),
            nn.Dropout(0.25), nn.Linear(128, 1)
        )
        self.temporal = nn.Sequential(
            nn.Conv1d(feature_dim, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=3, padding=1)
        )
        self.temporal_gate = nn.Parameter(torch.tensor(0.5))

    def forward(self, video):
        # video: [B,T,C,H,W]
        b, t, c, h, w = video.shape
        x = video.reshape(b * t, c, h, w)
        spatial = self.spatial(x)
        z = self.quantum_map(spatial)
        initial_logits = self.classifier(z).reshape(b, t)
        initial_prob = torch.sigmoid(initial_logits)

        entropy = -(
            initial_prob.clamp(1e-6, 1-1e-6) * torch.log(initial_prob.clamp(1e-6, 1-1e-6)) +
            (1-initial_prob).clamp(1e-6, 1) * torch.log((1-initial_prob).clamp(1e-6, 1))
        ) / math.log(2.0)
        video_uncertainty = entropy.mean(dim=1)
        uncertain_video = video_uncertainty > 0.45

        final_logits = initial_logits.clone()
        if uncertain_video.any():
            selected_z = z.reshape(b, t, -1)[uncertain_video]
            temporal_logits = self.temporal(selected_z.transpose(1, 2)).squeeze(1)
            selected_initial = initial_logits[uncertain_video]
            gate = torch.sigmoid(self.temporal_gate)
            final_logits[uncertain_video] = selected_initial + gate * temporal_logits

        final_prob = torch.sigmoid(final_logits)
        return {
            "initial_prob": initial_prob,
            "final_prob": final_prob,
            "entropy": entropy,
            "video_uncertainty": video_uncertainty,
            "uncertain_video": uncertain_video,
            "features": z.reshape(b, t, -1)
        }

model = EGLightEncoder(pretrained=True).to(DEVICE)
print("Hybrid feature dimension:", model.spatial.output_dim)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Weakly supervised video-level training

The official training set supplies video-level labels. We aggregate the highest-scoring sampled frames using a top-k mean and optimize binary cross-entropy at video level. The temporal branch is trained jointly on uncertain videos through the same video-level objective.

In [ ]:
def topk_video_score(frame_prob, ratio=TOPK_RATIO):
    k = max(1, int(math.ceil(frame_prob.shape[1] * ratio)))
    return frame_prob.topk(k, dim=1).values.mean(dim=1)

def video_loss(output, labels):
    scores = topk_video_score(output["final_prob"])
    return F.binary_cross_entropy(scores.clamp(1e-6, 1-1e-6), labels), scores

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def run_epoch(loader, training):
    model.train(training)
    losses, all_y, all_s = [], [], []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for batch in tqdm(loader, leave=False):
            frames = batch["frames"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            if training:
                optimizer.zero_grad(set_to_none=True)
            output = model(frames)
            loss, scores = video_loss(output, labels)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
            losses.append(loss.item())
            all_y.extend(labels.detach().cpu().numpy().tolist())
            all_s.extend(scores.detach().cpu().numpy().tolist())
    auc = roc_auc_score(all_y, all_s) if len(set(all_y)) > 1 else float("nan")
    pred = (np.asarray(all_s) >= 0.5).astype(int)
    return {
        "loss": float(np.mean(losses)),
        "auc": float(auc),
        "accuracy": accuracy_score(all_y, pred),
        "y": np.asarray(all_y), "scores": np.asarray(all_s)
    }

history = []
best_val_auc = -1
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step()
    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_auc": train_metrics["auc"],
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_accuracy": val_metrics["accuracy"]
    }
    history.append(row)
    print(row)
    if np.isfinite(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc:
        best_val_auc = val_metrics["auc"]
        best_state = copy.deepcopy(model.state_dict())

if best_state is not None:
    model.load_state_dict(best_state)

history_df = pd.DataFrame(history)
display(history_df)

## 8. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_df.epoch, history_df.train_loss, label="Train")
ax[0].plot(history_df.epoch, history_df.val_loss, label="Validation")
ax[0].set_title("Video-level loss"); ax[0].set_xlabel("Epoch"); ax[0].legend(); ax[0].grid(alpha=.25)
ax[1].plot(history_df.epoch, history_df.train_auc, label="Train AUC")
ax[1].plot(history_df.epoch, history_df.val_auc, label="Validation AUC")
ax[1].set_title("Video-level ROC-AUC"); ax[1].set_xlabel("Epoch"); ax[1].legend(); ax[1].grid(alpha=.25)
plt.tight_layout()
plt.show()

## 9. Validation-based confidence calibration

The confidence threshold is selected using validation videos only. Entropy determines whether temporal verification is activated; the anomaly decision threshold is selected by maximizing validation F1. Test labels and test temporal annotations are not used for calibration.

In [ ]:
@torch.no_grad()
def collect_video_predictions(loader):
    model.eval()
    rows = []
    for batch in tqdm(loader):
        frames = batch["frames"].to(DEVICE)
        out = model(frames)
        scores = topk_video_score(out["final_prob"]).cpu().numpy()
        entropy = out["entropy"].mean(dim=1).cpu().numpy()
        for i, path in enumerate(batch["path"]):
            rows.append({
                "path": path,
                "label": int(batch["label"][i].item()),
                "score": float(scores[i]),
                "entropy": float(entropy[i])
            })
    return pd.DataFrame(rows)

val_predictions = collect_video_predictions(val_loader)
threshold_candidates = np.linspace(0.05, 0.95, 91)
f1_values = [
    f1_score(val_predictions.label, (val_predictions.score >= th).astype(int), zero_division=0)
    for th in threshold_candidates
]
DECISION_THRESHOLD = float(threshold_candidates[int(np.argmax(f1_values))])
ENTROPY_THRESHOLD = float(np.quantile(val_predictions.entropy, 0.65))
print("Validation-calibrated anomaly threshold:", DECISION_THRESHOLD)
print("Validation-calibrated uncertainty threshold:", ENTROPY_THRESHOLD)

## 10. Temporal annotations and frame-level ground truth

The original annotation text commonly stores: video filename, anomaly category, start frame, end frame, and optional second start/end frame. The frame indices are interpreted as inclusive and converted to seconds using the video's measured FPS. If the annotation file is absent, video-level testing remains possible, but frame-level temporal metrics are skipped rather than fabricated.

In [ ]:
def load_temporal_annotations(annotation_file):
    annotations = {}
    if not Path(annotation_file).exists():
        print("Temporal annotation file not found; frame-level evaluation will be skipped.")
        return annotations
    with open(annotation_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 4:
                continue
            name = Path(parts[0]).name.lower()
            values = []
            for token in parts[2:]:
                try:
                    values.append(int(float(token)))
                except ValueError:
                    pass
            intervals = []
            for j in range(0, len(values)-1, 2):
                start, end = values[j], values[j+1]
                if start >= 0 and end >= start:
                    intervals.append((start, end))
            annotations[name] = intervals
    return annotations

temporal_annotations = load_temporal_annotations(TEMPORAL_ANNOTATION_FILE)
print("Videos with temporal annotations:", len(temporal_annotations))

def temporal_targets(video_name, sampled_frame_indices):
    intervals = temporal_annotations.get(Path(video_name).name.lower(), [])
    targets = []
    for frame_idx in sampled_frame_indices:
        targets.append(int(any(start <= int(frame_idx) <= end for start, end in intervals)))
    return np.asarray(targets, dtype=np.int64)

## 11. Fuzzy alert prioritisation

Inputs are final anomaly confidence, temporal persistence, and anomaly intensity. Triangular/trapezoidal-style membership functions and an explicit interpretable rule base produce Low, Medium, or High risk. The output is a prioritisation score, not a calibrated probability of harm.

In [ ]:
def tri_mf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    left = np.divide(x-a, b-a, out=np.zeros_like(x), where=(b-a)!=0)
    right = np.divide(c-x, c-b, out=np.zeros_like(x), where=(c-b)!=0)
    return np.clip(np.minimum(left, right), 0, 1)

def fuzzy_alert(confidence, persistence, intensity):
    c = float(np.clip(confidence, 0, 1))
    p = float(np.clip(persistence, 0, 1))
    i = float(np.clip(intensity, 0, 1))

    c_low, c_med, c_high = tri_mf(c, 0.0, 0.0, 0.55), tri_mf(c, 0.2, 0.55, 0.85), tri_mf(c, 0.6, 1.0, 1.0)
    p_low, p_med, p_high = tri_mf(p, 0.0, 0.0, 0.5), tri_mf(p, 0.2, 0.55, 0.85), tri_mf(p, 0.6, 1.0, 1.0)
    i_low, i_med, i_high = tri_mf(i, 0.0, 0.0, 0.5), tri_mf(i, 0.2, 0.55, 0.85), tri_mf(i, 0.6, 1.0, 1.0)

    # Interpretable rules:
    # High: strong confidence and persistent activity, or high confidence + high intensity.
    high = max(min(c_high, p_high), min(c_high, i_high, p_med))
    # Medium: moderate evidence sustained over time, or high confidence with moderate persistence.
    medium = max(min(c_med, p_med), min(c_high, p_med), min(c_med, i_high))
    # Low: weak confidence or weak/short-duration evidence.
    low = max(c_low, min(c_med, p_low), min(c_low, i_high))

    strengths = np.array([low, medium, high], dtype=float)
    if strengths.sum() <= 1e-8:
        strengths[0] = 1.0
    centers = np.array([0.2, 0.55, 0.9])
    crisp = float(np.sum(strengths * centers) / strengths.sum())
    category = "High Risk" if high >= max(medium, low) and high > 0 else (
        "Medium Risk" if medium >= low and medium > 0 else "Low Risk"
    )
    return crisp, category

def create_alerts(probabilities, frame_indices, fps, threshold=DECISION_THRESHOLD):
    flags = (np.asarray(probabilities) >= threshold).astype(np.int32)
    # Temporal decision smoothing: median filter over sampled time steps.
    if len(flags) >= 3:
        smooth = np.array([
            int(np.median(flags[max(0, j-1):min(len(flags), j+2)]) >= 0.5)
            for j in range(len(flags))
        ])
    else:
        smooth = flags.copy()
    alerts = []
    for j, flag in enumerate(smooth):
        if flag:
            persistence = float(np.mean(smooth[max(0,j-2):min(len(smooth),j+3)]))
            intensity = float(np.mean(probabilities[max(0,j-1):min(len(probabilities),j+2)]))
            confidence = float(probabilities[j])
            severity_score, severity = fuzzy_alert(confidence, persistence, intensity)
            timestamp = float(frame_indices[j]) / max(float(fps), 1e-6)
            alerts.append({
                "timestamp_seconds": timestamp,
                "frame_index": int(frame_indices[j]),
                "anomaly_confidence": confidence,
                "temporal_persistence": persistence,
                "severity_score": severity_score,
                "severity": severity
            })
    return alerts

## 12. Test-set inference, alert generation and evaluation

In [ ]:
@torch.no_grad()
def evaluate_test_set(loader):
    model.eval()
    video_rows, frame_rows, alert_rows = [], [], []
    for batch in tqdm(loader):
        frames = batch["frames"].to(DEVICE)
        out = model(frames)
        probs = out["final_prob"][0].cpu().numpy()
        init_probs = out["initial_prob"][0].cpu().numpy()
        indices = batch["frame_indices"][0].cpu().numpy()
        fps = float(batch["fps"][0].item())
        path = batch["path"][0]
        label = int(batch["label"][0].item())
        score = float(topk_video_score(out["final_prob"])[0].item())
        video_rows.append({
            "path": path, "video": Path(path).name, "label": label,
            "score": score, "mean_entropy": float(out["entropy"].mean().item()),
            "temporal_branch_activated": bool(out["uncertain_video"][0].item())
        })
        targets = temporal_targets(Path(path).name, indices)
        for k, (p, ip, fi, gt) in enumerate(zip(probs, init_probs, indices, targets)):
            frame_rows.append({
                "path": path, "video": Path(path).name,
                "frame_index": int(fi), "timestamp_seconds": float(fi)/max(fps,1e-6),
                "target": int(gt), "initial_score": float(ip), "score": float(p)
            })
        for alert in create_alerts(probs, indices, fps):
            alert_rows.append({"video": Path(path).name, **alert})
    return pd.DataFrame(video_rows), pd.DataFrame(frame_rows), pd.DataFrame(alert_rows)

test_video_results, test_frame_results, alert_table = evaluate_test_set(test_loader)
display(test_video_results.head())
display(alert_table.head())

In [ ]:
def safe_auc(y, s):
    return roc_auc_score(y, s) if len(np.unique(y)) > 1 else np.nan

video_pred = (test_video_results.score >= DECISION_THRESHOLD).astype(int)
video_metrics = {
    "Video ROC-AUC": safe_auc(test_video_results.label, test_video_results.score),
    "Video Average Precision": average_precision_score(test_video_results.label, test_video_results.score),
    "Video Accuracy": accuracy_score(test_video_results.label, video_pred),
    "Video Precision": precision_score(test_video_results.label, video_pred, zero_division=0),
    "Video Recall": recall_score(test_video_results.label, video_pred, zero_division=0),
    "Video F1": f1_score(test_video_results.label, video_pred, zero_division=0)
}
print("Video-level test metrics")
display(pd.DataFrame([video_metrics]).T.rename(columns={0:"Value"}))

if len(temporal_annotations) > 0 and len(test_frame_results) > 0 and test_frame_results.target.nunique() > 1:
    frame_pred = (test_frame_results.score >= DECISION_THRESHOLD).astype(int)
    frame_metrics = {
        "Frame ROC-AUC": safe_auc(test_frame_results.target, test_frame_results.score),
        "Frame Average Precision": average_precision_score(test_frame_results.target, test_frame_results.score),
        "Frame Accuracy": accuracy_score(test_frame_results.target, frame_pred),
        "Frame Precision": precision_score(test_frame_results.target, frame_pred, zero_division=0),
        "Frame Recall": recall_score(test_frame_results.target, frame_pred, zero_division=0),
        "Frame F1": f1_score(test_frame_results.target, frame_pred, zero_division=0)
    }
    print("Sampled-frame temporal test metrics")
    display(pd.DataFrame([frame_metrics]).T.rename(columns={0:"Value"}))
else:
    print("Frame-level metrics unavailable: temporal annotations are missing or sampled targets contain only one class.")

print("Video classification report")
print(classification_report(test_video_results.label, video_pred, target_names=["Normal","Anomalous"], zero_division=0))

## 13. Confusion matrix and ROC curves

In [ ]:
cm = confusion_matrix(test_video_results.label, video_pred, labels=[0,1])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0,1], ["Normal","Anomalous"])
ax.set_yticks([0,1], ["Normal","Anomalous"])
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Video-level normalized confusion matrix")
row_sums = cm.sum(axis=1, keepdims=True)
norm_cm = cm / np.maximum(row_sums, 1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{norm_cm[i,j]:.3f}", ha="center", va="center")
plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

if test_video_results.label.nunique() > 1:
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(test_video_results.label, test_video_results.score)
    plt.figure(figsize=(5.5,4.5))
    plt.plot(fpr, tpr, label=f"AUC = {safe_auc(test_video_results.label, test_video_results.score):.4f}")
    plt.plot([0,1],[0,1],"--",linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("Video-level ROC curve"); plt.legend(); plt.grid(alpha=.25); plt.tight_layout(); plt.show()

## 14. Edge optimisation: structured channel pruning and INT8 dynamic quantization

The pruning routine below applies L1 structured channel masks to convolution layers and removes the pruning reparameterization. This creates zeroed channels but does not physically shrink tensor dimensions; actual architecture slimming requires rebuilding the affected layers. Dynamic INT8 quantization is applied to supported Linear layers on CPU. Convolutional INT8 acceleration depends on the target runtime/backend and is not silently claimed here.

In [ ]:
def apply_structured_pruning(model, amount=0.10):
    try:
        import torch.nn.utils.prune as prune
        convs = [m for m in model.modules() if isinstance(m, nn.Conv2d)]
        for layer in convs:
            if layer.out_channels > 1:
                prune.ln_structured(layer, name="weight", amount=amount, n=1, dim=0)
                prune.remove(layer, "weight")
        return len(convs)
    except Exception as e:
        print("Structured pruning could not be applied:", repr(e))
        return 0

def model_size_mb(model):
    import io
    buffer = io.BytesIO()
    torch.save(model.state_dict(), buffer)
    return buffer.getbuffer().nbytes / (1024**2)

baseline_size = model_size_mb(model)
pruned_conv_count = apply_structured_pruning(model, amount=0.10)
pruned_size = model_size_mb(model)
print("Convolution layers processed:", pruned_conv_count)
print(f"State-dict size before pruning: {baseline_size:.2f} MB")
print(f"State-dict size after pruning:  {pruned_size:.2f} MB")

cpu_model = copy.deepcopy(model).to("cpu").eval()
if QUANTIZE_AFTER_TRAINING:
    try:
        quantized_model = torch.ao.quantization.quantize_dynamic(
            cpu_model, {nn.Linear}, dtype=torch.qint8
        )
        print(f"Dynamic INT8 model state size: {model_size_mb(quantized_model):.2f} MB")
    except Exception as e:
        quantized_model = cpu_model
        print("Dynamic quantization unavailable; using FP32 CPU model.", repr(e))
else:
    quantized_model = cpu_model

## 15. Latency, throughput and memory profiling

In [ ]:
def benchmark_model(model_to_test, device, warmup=5, runs=30):
    model_to_test = model_to_test.to(device).eval()
    x = torch.randn(1, NUM_FRAMES, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    with torch.no_grad():
        for _ in range(warmup):
            _ = model_to_test(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times = []
        for _ in range(runs):
            start = time.perf_counter()
            _ = model_to_test(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - start) * 1000)
    return {
        "device": str(device),
        "mean_latency_ms_per_video": float(np.mean(times)),
        "median_latency_ms_per_video": float(np.median(times)),
        "videos_per_second": float(1000 / np.mean(times)),
        "frames_per_second": float(NUM_FRAMES * 1000 / np.mean(times))
    }

benchmark_rows = []
benchmark_rows.append(benchmark_model(model, DEVICE))
benchmark_rows.append(benchmark_model(quantized_model, torch.device("cpu")))
benchmark_df = pd.DataFrame(benchmark_rows)
display(benchmark_df)

if torch.cuda.is_available():
    print("Peak CUDA allocated memory (MB):", round(torch.cuda.max_memory_allocated() / 1024**2, 2))
else:
    print("CUDA memory profiling skipped: CUDA is not available.")

## 16. Real-time inference interface

This function accepts a video path, applies the same uniform sampling and preprocessing, runs the trained model, and returns timestamped severity-prioritised alerts. For continuous streams, adapt the sampler to use a rolling frame buffer and maintain temporal context between windows.

In [ ]:
@torch.no_grad()
def predict_video(video_path, model_to_use=model, device=DEVICE):
    model_to_use.eval()
    frames, indices, fps, total = read_uniform_frames(video_path)
    tensor = VideoTransform(train=False)(frames).unsqueeze(0).to(device)
    out = model_to_use(tensor)
    probs = out["final_prob"][0].detach().cpu().numpy()
    score = float(topk_video_score(out["final_prob"])[0].item())
    alerts = create_alerts(probs, indices, fps)
    return {
        "video": str(video_path),
        "video_anomaly_score": score,
        "video_prediction": "Anomalous" if score >= DECISION_THRESHOLD else "Normal",
        "uncertainty": float(out["entropy"].mean().item()),
        "temporal_verification_activated": bool(out["uncertain_video"][0].item()),
        "alerts": alerts
    }

# Example:
# result = predict_video("/path/to/a/surveillance_clip.mp4")
# print(json.dumps(result, indent=2))

## 17. Save trained weights and evaluation outputs

In [ ]:
# Save only when the user chooses to run this cell.
OUTPUT_DIR = Path("Periyaswamy sir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "decision_threshold": DECISION_THRESHOLD,
    "entropy_threshold": ENTROPY_THRESHOLD,
    "image_size": IMAGE_SIZE,
    "num_frames": NUM_FRAMES,
    "feature_dim": FEATURE_DIM
}, OUTPUT_DIR / "EG_LightEncoder_UCF_Crime.pth")

test_video_results.to_csv(OUTPUT_DIR / "test_video_predictions.csv", index=False)
test_frame_results.to_csv(OUTPUT_DIR / "test_sampled_frame_predictions.csv", index=False)
alert_table.to_csv(OUTPUT_DIR / "prioritised_alerts.csv", index=False)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())

## 18. Reproducibility checklist and interpretation

- Confirm that every split entry resolves to the intended video and inspect the printed class counts.
- Keep the official test partition untouched during model selection and threshold calibration.
- The notebook's validation split is drawn only from the official training videos.
- Test frame metrics are computed at the uniformly sampled frame locations, not at every original video frame.
- Temporal labels are read from the supplied annotation file; missing annotations are not synthesized.
- Report the exact hardware, input frame count, image resolution, batch size, and precision when reporting latency.
- Structured pruning masks channels but does not guarantee smaller tensor shapes or faster inference. For deployment-specific speedups, rebuild the pruned network and benchmark on the actual edge device.
- The default timm pretrained weights may need an internet connection on first use. If unavailable, the notebook falls back to random initialization; for a meaningful experiment, use pretrained weights or explicitly document the initialization.
- This is a reproducible research implementation scaffold. Validate the selected timm model variants, dataset paths, annotation alignment, and edge-runtime support on the target environment before claiming final results.